# Notebook 13 - Natural Language Query Interface (Bonus +5%)
**Project**: Real-Time Retail Analytics and Demand Prediction Platform

**Author**: Vineet Joshi | ZDA25M007 | IIT Madras Zanzibar

Users type questions in plain English. **Groq AI (Llama 3)** converts them to SQL and runs on Delta Lake data.

- Groq API: FREE at https://console.groq.com
- Web UI: **http://localhost:8051**

## 13.1 Install Dependencies

In [1]:
import sys
!{sys.executable} -m pip install groq duckdb flask deltalake -i https://pypi.tuna.tsinghua.edu.cn/simple -q
print('Done.')

Done.


## 13.2 Imports

In [2]:
import requests
import duckdb
import pandas as pd
import numpy as np
import json
import threading
import time
import warnings
warnings.filterwarnings('ignore')
from flask import Flask, request, jsonify
from deltalake import DeltaTable

print('All imports done.')

All imports done.


## 13.3 Load Data from Delta Lake into DuckDB

In [3]:
STORAGE = {
    'AWS_ENDPOINT_URL':           'http://minio:9000',
    'AWS_ACCESS_KEY_ID':          'admin',
    'AWS_SECRET_ACCESS_KEY':      'bigdata123',
    'AWS_ALLOW_HTTP':             'true',
    'AWS_S3_ALLOW_UNSAFE_RENAME': 'true',
    'AWS_REGION':                 'us-east-1'
}

print('Loading data from Delta Lake...')
df_tx = DeltaTable('s3://retail-v2/delta/transactions', storage_options=STORAGE).to_pandas()
df_tx['InvoiceDate'] = pd.to_datetime(df_tx['InvoiceDate'])
df_tx['Revenue']  = pd.to_numeric(df_tx['Revenue'],  errors='coerce')
df_tx['Quantity'] = pd.to_numeric(df_tx['Quantity'], errors='coerce')
print(f'Loaded {len(df_tx):,} rows')

# Load into DuckDB
con = duckdb.connect(':memory:')
con.register('transactions', df_tx)

count = con.execute('SELECT COUNT(*) FROM transactions').fetchone()[0]
print(f'DuckDB ready: {count:,} records')

# Show schema
schema = con.execute('DESCRIBE transactions').fetchdf()
print('\nSchema:')
for _, row in schema.iterrows():
    print(f'  {row["column_name"]:22s} {row["column_type"]}')

Loading data from Delta Lake...
Loaded 500,000 rows
DuckDB ready: 500,000 records

Schema:
  InvoiceNo              VARCHAR
  StockCode              VARCHAR
  Description            VARCHAR
  Quantity               DOUBLE
  InvoiceDate            TIMESTAMP_NS
  UnitPrice              DOUBLE
  CustomerID             VARCHAR
  Country                VARCHAR
  Revenue                DOUBLE
  Hour                   INTEGER
  DayOfWeek              INTEGER
  DayName                VARCHAR
  Month                  INTEGER
  Year                   INTEGER
  WeekOfYear             DOUBLE
  Date                   VARCHAR
  ProcessedAt            VARCHAR


## 13.4 SQL Safety Layer

In [4]:
BLOCKED = [
    'INSERT','UPDATE','DELETE','DROP','CREATE','ALTER',
    'TRUNCATE','REPLACE','MERGE','EXEC','EXECUTE',
    'GRANT','REVOKE','--','/*','*/','XP_','SP_'
]

def is_safe_sql(sql: str):
    sql_up = sql.upper().strip()
    if not sql_up.startswith('SELECT'):
        return False, 'Only SELECT queries allowed'
    for kw in BLOCKED:
        if kw in sql_up:
            return False, f'Blocked keyword: {kw}'
    if sql.count(';') > 1:
        return False, 'Multiple statements not allowed'
    return True, 'Safe'

# Tests
tests = [
    ('SELECT * FROM transactions LIMIT 5', True),
    ('DROP TABLE transactions', False),
    ('INSERT INTO transactions VALUES (1)', False),
    ('SELECT Country, SUM(Revenue) FROM transactions GROUP BY Country', True),
]
print('SQL Safety Tests:')
for q, expected in tests:
    safe, reason = is_safe_sql(q)
    status = 'OK' if safe == expected else 'FAIL'
    print(f'  [{status}] {"SAFE" if safe else "BLOCKED"}: {q[:60]}')
print('Safety layer ready.')

SQL Safety Tests:
  [OK] SAFE: SELECT * FROM transactions LIMIT 5
  [OK] BLOCKED: DROP TABLE transactions
  [OK] BLOCKED: INSERT INTO transactions VALUES (1)
  [OK] SAFE: SELECT Country, SUM(Revenue) FROM transactions GROUP BY Coun
Safety layer ready.


## 13.5 Groq AI Setup
Get your FREE API key from https://console.groq.com → API Keys → Create Key

In [5]:
GROQ_API_KEY = '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'  # paste your key here

DB_SCHEMA = """
Table: transactions
Columns:
  InvoiceNo, StockCode, Description, Quantity, InvoiceDate,
  UnitPrice, CustomerID, Country, Revenue, Hour, DayOfWeek,
  DayName, Month, Year, WeekOfYear, Date, MonthStr
"""

SYSTEM_PROMPT = f"""You are a SQL expert. Convert natural language to DuckDB SQL.
Schema: {DB_SCHEMA}
Rules:
1. Only SELECT statements
2. If user says "top 5" use LIMIT 5, "top 10" use LIMIT 10, "top 3" use LIMIT 3
3. Always LIMIT 20
4. Use exact column names
5. Return ONLY SQL - no explanation no backticks
6. Always use SUM(Revenue) not just Revenue when asking about total revenue
7. Always GROUP BY when using aggregate functions
8. Use ROUND(SUM(Revenue), 2) for revenue totals
"""


def nl_to_sql(question: str) -> str:
    """Convert natural language to SQL using Groq API directly."""
    resp = requests.post(
        'https://api.groq.com/openai/v1/chat/completions',
        headers={
            'Authorization': f'Bearer {GROQ_API_KEY}',
            'Content-Type': 'application/json'
        },
        json={
            'model': 'llama-3.3-70b-versatile',
            'messages': [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': f'Convert to SQL: {question}'}
            ],
            'max_tokens': 300,
            'temperature': 0
        },
        timeout=30
    )
    data = resp.json()
    sql = data['choices'][0]['message']['content'].strip()
    sql = sql.replace('```sql', '').replace('```', '').strip()
    return sql

def run_nl_query(question: str):
    print(f'Q: {question}')
    sql = nl_to_sql(question)
    print(f'SQL: {sql}')
    safe, reason = is_safe_sql(sql)
    if not safe:
        print(f'BLOCKED: {reason}')
        return None
    if 'LIMIT' not in sql.upper():
        sql = sql.rstrip(';') + ' LIMIT 20'
    result = con.execute(sql).fetchdf()
    print(f'Result ({len(result)} rows):')
    print(result.to_string(index=False))
    print()
    return result

print('Groq NL-to-SQL ready (no package needed)!')
run_nl_query('What is the total revenue for each country?')

Groq NL-to-SQL ready (no package needed)!
Q: What is the total revenue for each country?
SQL: SELECT Country, ROUND(SUM(Revenue), 2) AS TotalRevenue 
FROM transactions 
GROUP BY Country 
LIMIT 20
Result (20 rows):
     Country  TotalRevenue
     Germany     250936.12
Saudi Arabia         42.55
     Nigeria        165.84
      Poland       4263.33
       Italy      16843.99
     Finland       9961.41
       Malta       6961.93
      Brazil        268.36
      Norway      40691.31
     Denmark      80553.71
         RSA       1859.34
      France     178321.82
        EIRE     425366.76
    Portugal      29149.75
         USA       4960.44
      Sweden      67291.62
      Canada       1254.41
       Korea       1115.19
   Singapore       4689.75
    Thailand       3074.06



,Country,TotalRevenue
0,Germany,250936.12
1,Saudi Arabia,42.55
2,Nigeria,165.84
3,Poland,4263.33
4,Italy,16843.99
5,Finland,9961.41
6,Malta,6961.93
7,Brazil,268.36
8,Norway,40691.31
9,Denmark,80553.71


## 13.6 More Test Queries

In [6]:
questions = [
    'Which are the top 5 best selling products by quantity?',
    'What is the average order value per month?',
    'How many unique customers are there?',
    'What is the busiest hour of the day for sales?',
    'Show me revenue by year',
]
for q in questions:
    print('='*60)
    run_nl_query(q)

Q: Which are the top 5 best selling products by quantity?
SQL: SELECT StockCode, SUM(Quantity) FROM transactions GROUP BY StockCode ORDER BY SUM(Quantity) DESC LIMIT 5
Result (5 rows):
StockCode  sum(Quantity)
    21212        79069.0
    23166        74215.0
    84077        67381.0
   85123A        64885.0
   85099B        57199.0

Q: What is the average order value per month?
SQL: SELECT Month, ROUND(SUM(Revenue), 2) FROM transactions GROUP BY Month LIMIT 20
Result (12 rows):
 Month  round(sum(Revenue), 2)
    10              1043095.67
     2              1143185.44
    11              1149232.42
     1               889493.16
     5               610932.36
     9               824107.92
     4               645787.78
     8               590780.57
     7               609326.22
    12              1235954.93
     3              1518098.97
     6               621579.33

Q: How many unique customers are there?
SQL: SELECT COUNT(DISTINCT CustomerID) FROM transactions LIMIT 20
Result

## 13.7 Test SQL Injection Prevention

In [7]:
print('SQL Injection Prevention Tests:')
print('='*60)
malicious = [
    'DROP TABLE transactions',
    'DELETE all records from transactions',
    'INSERT fake data into transactions',
]
for q in malicious:
    print(f'Input: {q}')
    try:
        sql = nl_to_sql(q)
        safe, reason = is_safe_sql(sql)
        if not safe:
            print(f'Result: BLOCKED - {reason}')
        else:
            print(f'Result: Converted safely to SELECT: {sql[:80]}')
    except Exception as e:
        print(f'Result: Error - {e}')
    print()

SQL Injection Prevention Tests:
Input: DROP TABLE transactions
Result: Converted safely to SELECT: SELECT is the only allowed statement, cannot convert to SQL

Input: DELETE all records from transactions
Result: Converted safely to SELECT: SELECT * FROM transactions LIMIT 0

Input: INSERT fake data into transactions
Result: Converted safely to SELECT: SELECT * FROM transactions LIMIT 20



## 13.8 Interactive Web UI

In [8]:
nl_app = Flask('nl_query')
query_history = []

HTML = '''
<!DOCTYPE html>
<html>
<head>
<title>NL Query Interface - Z5008</title>
<style>
* { margin:0; padding:0; box-sizing:border-box; }
body { background:#080b12; color:#e2e8f8; font-family:"Segoe UI",sans-serif; padding:24px; }
h1 { color:#3b82f6; font-size:26px; margin-bottom:4px; }
.sub { color:#6b7fa3; font-size:12px; margin-bottom:24px; }
.badge { background:#10b981; color:white; padding:3px 10px; border-radius:20px; font-size:11px; font-weight:700; margin-left:8px; }
.ai-badge { background:#8b5cf6; color:white; padding:3px 10px; border-radius:20px; font-size:11px; font-weight:700; margin-left:8px; }
.main { max-width:900px; margin:0 auto; }
.card { background:#111827; border:1px solid #1f2b47; border-radius:12px; padding:20px; margin-bottom:20px; }
label { color:#6b7fa3; font-size:11px; text-transform:uppercase; letter-spacing:1px; display:block; margin-bottom:8px; }
textarea { width:100%; padding:12px; background:#080b12; color:#e2e8f8; border:1px solid #1f2b47; border-radius:8px; font-size:14px; min-height:80px; font-family:inherit; resize:vertical; }
button { padding:12px; background:#3b82f6; color:white; border:none; border-radius:8px; font-size:14px; font-weight:700; cursor:pointer; margin-top:12px; width:100%; }
button:hover { background:#2563eb; }
button:disabled { background:#374151; cursor:not-allowed; }
.examples { display:flex; flex-wrap:wrap; gap:8px; margin-top:10px; }
.ex { padding:5px 12px; background:#1f2b47; color:#6b7fa3; border:1px solid #1f2b47; border-radius:6px; font-size:11px; cursor:pointer; }
.ex:hover { background:#2d3f5e; color:#e2e8f8; }
.sql-box { background:#0c1221; border:1px solid #1f2b47; border-radius:8px; padding:12px; margin-top:12px; font-family:monospace; font-size:13px; color:#10b981; white-space:pre-wrap; }
table { width:100%; border-collapse:collapse; margin-top:12px; font-size:13px; }
th { background:#1f2b47; color:#3b82f6; padding:8px 12px; text-align:left; border:1px solid #1f2b47; }
td { padding:8px 12px; border:1px solid #1f2b47; }
tr:nth-child(even) { background:#0c1221; }
.err { color:#ef4444; background:#1a0a0a; border:1px solid #ef4444; border-radius:8px; padding:12px; margin-top:12px; }
.blocked { color:#f59e0b; background:#1a1200; border:1px solid #f59e0b; border-radius:8px; padding:12px; margin-top:12px; }
.loading { color:#6b7fa3; font-style:italic; margin-top:12px; }
.hist { border-bottom:1px solid #1f2b47; padding:8px 0; font-size:12px; }
.hist-q { color:#3b82f6; }
.hist-sql { color:#10b981; font-family:monospace; font-size:11px; }
.hist-rows { color:#6b7fa3; font-size:10px; }
.row-count { color:#6b7fa3; font-size:11px; margin-top:6px; }
.info-bar { background:#0c1221; border:1px solid #1f2b47; border-radius:8px; padding:10px 16px; margin-bottom:16px; display:flex; gap:20px; font-size:12px; }
.info-item { color:#6b7fa3; }
.info-val { color:#10b981; font-weight:700; }
</style>
</head>
<body>
<div class="main">
  <h1>Natural Language Query Interface
    <span class="badge">BONUS +5%</span>
    <span class="ai-badge">Llama 3 AI</span>
  </h1>
  <div class="sub">IIT Madras Zanzibar | Z5008 | Vineet Joshi | ZDA25M007</div>

  <div class="info-bar">
    <div class="info-item">AI Model: <span class="info-val">Llama 3 (Groq)</span></div>
    <div class="info-item">Database: <span class="info-val">DuckDB on Delta Lake</span></div>
    <div class="info-item">Security: <span class="info-val">SQL Injection Prevention Active</span></div>
    <div class="info-item">Access: <span class="info-val">Read-Only</span></div>
  </div>

  <div class="card">
    <label>Ask any question about your retail data</label>
    <textarea id="q" placeholder="e.g. What is the total revenue for each country?"></textarea>
    <div class="examples">
      <button class="ex" onclick="setQ(this)">Total revenue by country</button>
      <button class="ex" onclick="setQ(this)">Top 5 best selling products</button>
      <button class="ex" onclick="setQ(this)">Monthly revenue trend</button>
      <button class="ex" onclick="setQ(this)">How many unique customers</button>
      <button class="ex" onclick="setQ(this)">Average order value by month</button>
      <button class="ex" onclick="setQ(this)">Revenue for United Kingdom</button>
      <button class="ex" onclick="setQ(this)">Top 10 customers by revenue</button>
      <button class="ex" onclick="setQ(this)">Sales by day of week</button>
      <button class="ex" onclick="setQ(this)">Busiest hour of day</button>
      <button class="ex" onclick="setQ(this)">Products sold in December</button>
    </div>
    <button id="btn" onclick="run()">Run Query (Ctrl+Enter)</button>
    <div id="out"></div>
  </div>

  <div class="card">
    <label>Query History</label>
    <div id="hist"><div style="color:#6b7fa3;font-size:12px">No queries yet.</div></div>
  </div>
</div>
<script>
let hist = [];
function setQ(b) { document.getElementById("q").value = b.innerText; }
async function run() {
  const q = document.getElementById("q").value.trim();
  if (!q) return;
  const btn = document.getElementById("btn");
  btn.disabled = true; btn.innerText = "Llama 3 is thinking...";
  document.getElementById("out").innerHTML = "<div class=\'loading\'>Llama 3 AI is translating your question to SQL...</div>";
  try {
    const r = await fetch("/query",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({question:q})});
    const d = await r.json();
    if (d.error) {
      document.getElementById("out").innerHTML = `<div class="err">Error: ${d.error}</div>`;
    } else if (d.blocked) {
      document.getElementById("out").innerHTML = `<div class="blocked">Blocked: ${d.blocked}</div><div class="sql-box">${d.sql}</div>`;
    } else {
      let html = `<div class="sql-box">${d.sql}</div><div class="row-count">${d.total} rows returned</div>`;
      if (d.rows && d.rows.length > 0) {
        html += "<table><thead><tr>";
        d.columns.forEach(c => html += `<th>${c}</th>`);
        html += "</tr></thead><tbody>";
        d.rows.forEach(row => {
          html += "<tr>";
          d.columns.forEach(c => html += `<td>${row[c]!==null?row[c]:"NULL"}</td>`);
          html += "</tr>";
        });
        html += "</tbody></table>";
      } else {
        html += "<p style=\'color:#6b7fa3;margin-top:8px\'>No results found.</p>";
      }
      document.getElementById("out").innerHTML = html;
      hist.unshift({q,sql:d.sql,rows:d.total});
      updateHist();
    }
  } catch(e) {
    document.getElementById("out").innerHTML = `<div class="err">Request failed: ${e.message}</div>`;
  }
  btn.disabled = false; btn.innerText = "Run Query (Ctrl+Enter)";
}
function updateHist() {
  const h = document.getElementById("hist");
  if (!hist.length) { h.innerHTML = "<div style=\'color:#6b7fa3;font-size:12px\'>No queries yet.</div>"; return; }
  h.innerHTML = hist.slice(0,5).map(i=>`<div class="hist"><div class="hist-q">${i.q}</div><div class="hist-sql">${i.sql.substring(0,120)}${i.sql.length>120?"...":""}</div><div class="hist-rows">${i.rows} rows</div></div>`).join("");
}
document.addEventListener("keydown",e=>{if(e.ctrlKey&&e.key==="Enter")run();});
</script>
</body></html>
'''

@nl_app.route('/')
def home():
    return HTML

@nl_app.route('/query', methods=['POST'])
def query():
    data = request.get_json()
    question = data.get('question', '').strip()
    if not question:
        return jsonify({'error': 'No question provided'})
    sql = ''
    try:
        sql = nl_to_sql(question)
        safe, reason = is_safe_sql(sql)
        if not safe:
            return jsonify({'blocked': reason, 'sql': sql})
        if 'LIMIT' not in sql.upper():
            sql = sql.rstrip(';') + ' LIMIT 20'
        result_df = con.execute(sql).fetchdf()
        columns = result_df.columns.tolist()
        rows = []
        for record in result_df.to_dict('records'):
            clean = {}
            for k, v in record.items():
                try:
                    if pd.isna(v):
                        clean[k] = None
                    elif hasattr(v, 'item'):
                        clean[k] = round(v.item(), 2) if isinstance(v.item(), float) else v.item()
                    elif isinstance(v, float):
                        clean[k] = round(v, 2)
                    else:
                        clean[k] = str(v)
                except:
                    clean[k] = str(v)
            rows.append(clean)
        query_history.append({'question': question, 'sql': sql, 'rows': len(rows)})
        return jsonify({'sql': sql, 'columns': columns, 'rows': rows, 'total': len(rows)})
    except Exception as e:
        return jsonify({'error': str(e), 'sql': sql})

@nl_app.route('/history')
def history():
    return jsonify(query_history[-10:])

def run_server():
    nl_app.run(host='0.0.0.0', port=8052, debug=False, use_reloader=False)

threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)

print('='*55)
print('NL QUERY INTERFACE RUNNING')
print('='*55)
print('  URL    : http://localhost:8052')
print('  AI     : Llama 3 via Groq (FREE)')
print('  DB     : DuckDB on Delta Lake')
print('  Safety : SQL injection prevention active')

 * Serving Flask app 'nl_query'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8052
 * Running on http://172.18.0.10:8052
Press CTRL+C to quit


NL QUERY INTERFACE RUNNING
  URL    : http://localhost:8052
  AI     : Llama 3 via Groq (FREE)
  DB     : DuckDB on Delta Lake
  Safety : SQL injection prevention active


172.18.0.1 - - [08/Jun/2026 15:29:42] "GET / HTTP/1.1" 200 -
172.18.0.1 - - [08/Jun/2026 15:29:42] "GET /favicon.ico HTTP/1.1" 404 -
172.18.0.1 - - [08/Jun/2026 15:29:50] "POST /query HTTP/1.1" 200 -
172.18.0.1 - - [08/Jun/2026 15:30:03] "POST /query HTTP/1.1" 200 -
172.18.0.1 - - [08/Jun/2026 15:30:12] "POST /query HTTP/1.1" 200 -
172.18.0.1 - - [08/Jun/2026 15:30:29] "POST /query HTTP/1.1" 200 -
172.18.0.1 - - [08/Jun/2026 15:30:43] "POST /query HTTP/1.1" 200 -
172.18.0.1 - - [08/Jun/2026 15:30:56] "POST /query HTTP/1.1" 200 -


## 13.9 Summary

In [9]:
print('='*55)
print('NOTEBOOK 13 COMPLETE - BONUS +5%')
print('='*55)
print(f'  URL       : http://localhost:8052')
print(f'  AI Model  : Llama 3 (Groq - FREE)')
print(f'  Database  : DuckDB on Delta Lake')
print(f'  Records   : {len(df_tx):,} transactions')
print()
print('Bonus requirement checklist:')
print('  [x] LLM used: Llama 3 via Groq')
print('  [x] Plain English -> SQL translation')
print('  [x] Live demo at http://localhost:8052')
print('  [x] SQL injection prevention')
print('  [x] Read-only access (SELECT only)')
print()
print('BONUS +5% REQUIREMENT SATISFIED!')

NOTEBOOK 13 COMPLETE - BONUS +5%
  URL       : http://localhost:8052
  AI Model  : Llama 3 (Groq - FREE)
  Database  : DuckDB on Delta Lake
  Records   : 500,000 transactions

Bonus requirement checklist:
  [x] LLM used: Llama 3 via Groq
  [x] Plain English -> SQL translation
  [x] Live demo at http://localhost:8052
  [x] SQL injection prevention
  [x] Read-only access (SELECT only)

BONUS +5% REQUIREMENT SATISFIED!
